<div dir="rtl" style="text-align: right;">

#  جستجوی چند-برداری با ColBERT در وی‌ویت (Weaviate)

<div align="center">
  <img src="images/weaviate.png" alt="ColBERT Overview" width="10%">
</div>

در آزمایشگاه‌های قبلی با API وی‌ویت آشنا شدید و از یک vectorizer معمولی (`text2vec-ollama` / `text2vec-openai`) استفاده کردید که هر سند را به **یک بردار** تبدیل می‌کند. در این آزمایشگاه با روش دیگری از بازیابی آشنا می‌شوید: **late interaction** با مدل **ColBERT**، که هر سند را به‌جای یک بردار، به یک **مجموعه بردار** (یک بردار به‌ازای هر توکن) تبدیل می‌کند.

چون وی‌ویت (در تنظیمات فعلی `docker-compose.yml`) ماژول بومی برای ColBERT فعال ندارد، خودمان بردارها را با کتابخانه‌ی سبک `fastembed` تولید می‌کنیم و به‌صورت **self-provided multi-vector** به وی‌ویت می‌دهیم.

بزن بریم!

</div>

In [2]:
from IPython.display import HTML

# راست‌چین کردن سلول‌های متنی (Markdown) در این نوت‌بوک
HTML("""
<style>
  .jp-RenderedMarkdown, .rendered_html, .text_cell_render {
    direction: rtl;
    text-align: right;
  }
</style>
""")


<div dir="rtl" style="text-align: right;">

# فهرست مطالب
- [ ۱ - مقدمه‌ای بر ColBERT](#1)
  - [ ۱.۱ بارگذاری کتابخانه‌های لازم](#1-1)
  - [ ۱.۲ کلاینت وی‌ویت](#1-2)
- [ ۲ - تولید بردارهای ColBERT](#2)
  - [ ۲.۱ بارگذاری مدل با fastembed](#2-1)
  - [ ۲.۲ بردارسازی چند سند نمونه](#2-2)
- [ ۳ - ساخت Collection با پیکربندی self-provided](#3)
  - [ ۳.۱ تعریف Collection](#3-1)
  - [ ۳.۲ افزودن اسناد همراه با بردارهای ColBERT](#3-2)
- [ ۴ - جستجو با ColBERT (MaxSim)](#4)
  - [ ۴.۱ بردارسازی کوئری (query_embed)](#4-1)
  - [ ۴.۲ جستجو با near_vector](#4-2)
  - [ ۴.۳ مقایسه با جستجوی معنایی معمولی](#4-3)
- [ ۵ - جمع‌بندی](#5)

</div>

<div dir="rtl" style="text-align: right;">

<a id='1'></a>
## ۱ - مقدمه‌ای بر ColBERT
---

**ColBERT** («Contextualized Late Interaction over BERT») یک مدل بازیابی است که به‌جای فشرده کردن هر سند و کوئری در یک بردار (مثل مدل‌های embedding معمولی)، هر **توکن** را جداگانه بردارسازی می‌کند. شباهت بین یک کوئری و یک سند با **MaxSim** محاسبه می‌شود: برای هر بردار توکن کوئری، بیشترین شباهت با بردارهای توکن سند پیدا و جمع می‌شود.

به این تکنیک **late interaction** می‌گویند، چون تعامل بین کوئری و سند دیر (بعد از بردارسازی مستقل هرکدام) اتفاق می‌افتد؛ برخلاف مدل‌های dense معمولی که کل معنا را در یک بردار خلاصه می‌کنند و ممکن است جزئیات ظریف را از دست بدهند.

<a id='1-1'></a>
### ۱.۱ بارگذاری کتابخانه‌های لازم

`pip install weaviate-client fastembed tqdm`

سلول زیر را اجرا کنید تا کتابخانه‌های لازم بارگذاری شوند.

</div>

In [1]:
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import Filter, MetadataQuery
from weaviate.util import generate_uuid5
from fastembed import LateInteractionTextEmbedding
from tqdm import tqdm
import weaviate
import numpy as np


In [2]:
from utils import (
    suppress_subprocess_output,
    print_object_properties,
)


<div dir="rtl" style="text-align: right;">

<a id='1-2'></a>
### ۱.۲ کلاینت وی‌ویت

مثل قبل فرض می‌کنیم Weaviate با `docker-compose.yml` از قبل بالا آمده است، پس فقط کافیست وصل شویم.

</div>

In [4]:
with suppress_subprocess_output():
    client = weaviate.connect_to_local()

<div dir="rtl" style="text-align: right;">

<a id='2'></a>
## ۲ - تولید بردارهای ColBERT
---

<a id='2-1'></a>
### ۲.۱ بارگذاری مدل با fastembed

`fastembed` کتابخانه‌ای سبک و بدون نیاز به GPU است (بر پایه‌ی ONNX) که چند مدل ColBERT-مانند را پشتیبانی می‌کند. در این‌جا از مدل کلاسیک `colbert-ir/colbertv2.0` استفاده می‌کنیم؛ **توجه کنید این مدل فقط انگلیسی است** (نسخه‌ی فارسی/چندزبانه را در نوت‌بوک بعدی می‌بینیم).

اولین اجرا مدل را دانلود می‌کند (چند صد مگابایت)، پس ممکن است کمی طول بکشد.

</div>

In [6]:
colbert_model = LateInteractionTextEmbedding(model_name="colbert-ir/colbertv2.0")

<div dir="rtl" style="text-align: right;">

<a id='2-2'></a>
### ۲.۲ بردارسازی چند سند نمونه

بیایید یک دیتاست کوچک از چکیده‌های موضوعات فناوری بسازیم و آن‌ها را بردارسازی کنیم.

</div>

In [10]:
data = [
    {"title": "Vector Databases", "text": "Vector databases store high-dimensional embeddings and enable fast approximate nearest neighbor search for semantic retrieval."},
    {"title": "Late Interaction", "text": "Late interaction models like ColBERT represent each token with its own vector, enabling fine-grained token-level matching between queries and documents."},
    {"title": "Docker", "text": "Docker packages applications and their dependencies into portable containers that run consistently across environments."},
    {"title": "Transformers", "text": "The transformer architecture uses self-attention to model relationships between tokens in a sequence, powering modern language models."},
    {"title": "HNSW", "text": "Hierarchical Navigable Small World graphs are a popular indexing structure for fast approximate nearest neighbor search."},
    {"title": "RAG", "text": "Retrieval-Augmented Generation combines a retriever with a language model to ground answers in external documents."},
]

texts = [d["text"] for d in data]

# embed(...) یک generator برمی‌گرداند؛ هر خروجی یک آرایه‌ی (تعداد توکن × ابعاد) است
doc_vectors = list(colbert_model.embed(texts))

print("تعداد اسناد:", len(doc_vectors))
print("شکل بردار سند اول (تعداد توکن، ابعاد):", doc_vectors[0].shape)


تعداد اسناد: 6
شکل بردار سند اول (تعداد توکن، ابعاد): (22, 128)


In [18]:
doc_vectors[-1].shape

(20, 128)

<div dir="rtl" style="text-align: right;">

می‌بینید که برخلاف embedding معمولی (یک بردار ثابت مثلاً ۱۵۳۶ بعدی برای هر سند)، این‌جا **به تعداد توکن‌های سند، بردار** داریم — دقیقاً همان چیزی که در توضیحات ColBERT گفتیم.

</div>

<div dir="rtl" style="text-align: right;">

<a id='3'></a>
## ۳ - ساخت Collection با پیکربندی self-provided
---

<a id='3-1'></a>
### ۳.۱ تعریف Collection

چون خودمان بردارها را تولید می‌کنیم (نه وی‌ویت)، از `Configure.MultiVectors.self_provided` استفاده می‌کنیم. این با `Configure.NamedVectors.text2vec_ollama` که در آزمایشگاه‌های قبلی دیدید فرق دارد: آن‌جا وی‌ویت خودش بردارسازی می‌کرد، این‌جا فقط بردارهایی که به آن می‌دهیم را ایندکس می‌کند.

</div>

In [22]:
COLLECTION_NAME = "colbert_demo"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

collection = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=[
        Configure.MultiVectors.self_provided(
            name="colbert",  # نامی که برای درج و جستجو با آن به این بردار اشاره می‌کنیم
        ),
    ],
    properties=[
        Property(name="title", data_type=DataType.TEXT),
        Property(name="text", data_type=DataType.TEXT),
    ],
)

print(collection)


<weaviate.Collection config={
  "name": "Colbert_demo",
  "description": null,
  "generative_config": null,
  "inverted_index_config": {
    "bm25": {
      "b": 0.75,
      "k1": 1.2
    },
    "cleanup_interval_seconds": 60,
    "index_null_state": false,
    "index_property_length": false,
    "index_timestamps": false,
    "stopwords": {
      "preset": "en",
      "additions": null,
      "removals": null
    },
    "stopword_presets": null
  },
  "multi_tenancy_config": {
    "enabled": false,
    "auto_tenant_creation": false,
    "auto_tenant_activation": false
  },
  "object_ttl_config": null,
  "properties": [
    {
      "name": "title",
      "description": null,
      "data_type": "text",
      "index_filterable": true,
      "index_range_filters": false,
      "index_searchable": true,
      "nested_properties": null,
      "text_analyzer": null,
      "tokenization": "word",
      "vectorizer_config": null,
      "vectorizer": null,
      "vectorizer_configs": {}
    },


<div dir="rtl" style="text-align: right;">

<a id='3-2'></a>
### ۳.۲ افزودن اسناد همراه با بردارهای ColBERT

برخلاف قبل که فقط `properties` را پاس می‌دادیم و بردارسازی خودکار بود، این‌جا باید بردار هر آبجکت را هم صریحاً بدهیم؛ به‌صورت یک دیکشنری با کلید همان نامی که در `vector_config` تعریف کردیم (`"colbert"`) و مقدار، لیستی از بردارهای توکن (multi-vector).

</div>

In [24]:
with collection.batch.fixed_size(batch_size=1, concurrent_requests=1) as batch:
    for document, vec in tqdm(list(zip(data, doc_vectors))):
        uuid = generate_uuid5(document)
        batch.add_object(
            properties=document,
            uuid=uuid,
            vector={"colbert": vec.tolist()},  # multi-vector: لیستی از لیست‌های عددی
        )

print(len(collection))


100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 55.71it/s]

6


<div dir="rtl" style="text-align: right;">

<a id='4'></a>
## ۴ - جستجو با ColBERT (MaxSim)
---

<a id='4-1'></a>
### ۴.۱ بردارسازی کوئری (query_embed)

نکته‌ی مهم: ColBERT برای کوئری و سند از **دو تابع متفاوت** استفاده می‌کند. کوئری‌ها معمولاً کوتاه‌تر هستند و مدل با یک تکنیک به نام *query augmentation* (padding با توکن‌های [MASK]) آن‌ها را غنی‌تر می‌کند. به همین دلیل به‌جای `embed()`، از `query_embed()` استفاده می‌کنیم.

</div>

In [26]:
query = "How do vector databases find similar items quickly?"

query_vector = list(colbert_model.query_embed(query))[0]
print("شکل بردار کوئری (تعداد توکن، ابعاد):", query_vector.shape)


شکل بردار کوئری (تعداد توکن، ابعاد): (32, 128)


<div dir="rtl" style="text-align: right;">

<a id='4-2'></a>
### ۴.۲ جستجو با near_vector

حالا بردار چند-توکنی کوئری را به `near_vector` می‌دهیم. وی‌ویت به‌صورت خودکار امتیاز **MaxSim** را بین بردارهای کوئری و بردارهای هر سند محاسبه می‌کند (چون Collection ما با یک بردار multi-vector ساخته شده).

</div>

In [28]:
result = collection.query.near_vector(
    near_vector={"colbert": query_vector.tolist()},
    target_vector="colbert",
    limit=3,
    return_metadata=MetadataQuery(distance=True),
)

for obj in result.objects:
    print(f"فاصله: {obj.metadata.distance:.4f}")
    print_object_properties(obj.properties)


فاصله: -20.4606
title: Vector Databases
text: Vector databases store high-dimensional embeddings and enable fast approximate nearest neighbor search for semantic retrieval.

فاصله: -12.9342
title: HNSW
text: Hierarchical Navigable Small World graphs are a popular indexing structure for fast approximate nearest neighbor search.

فاصله: -11.4797
title: Late Interaction
text: Late interaction models like ColBERT represent each token with its own vector, enabling fine-grained token-level matching between queries and documents.



<div dir="rtl" style="text-align: right;">

<a id='4-3'></a>
### ۴.۳ مقایسه با جستجوی معنایی معمولی

نکته‌ی آموزشی: چون ColBERT معنای هر **توکن** را جداگانه نگه می‌دارد (نه یک بردار خلاصه‌شده)، معمولاً روی کوئری‌های طولانی، دقیق و دارای اصطلاحات نادر (out-of-domain) بهتر از embedding تک‌برداری معمولی عمل می‌کند — هرچند حجم ذخیره‌سازی و هزینه‌ی محاسباتی‌اش هم بیشتر است. به همین دلیل در عمل معمولاً به‌عنوان مرحله‌ی **rerank** بعد از یک بازیابی سریع اولیه (مثلاً با BM25 یا embedding معمولی) استفاده می‌شود، نه به‌عنوان تنها روش بازیابی.

</div>

<div dir="rtl" style="text-align: right;">

<a id='5'></a>
## ۵ - جمع‌بندی
---

در این آزمایشگاه یاد گرفتید:
- مفهوم late interaction و MaxSim در ColBERT چیست
- چطور با `fastembed` بردارهای multi-vector تولید کنید
- چطور یک Collection با پیکربندی `self_provided` multi-vector در وی‌ویت بسازید
- چطور اسناد و کوئری‌ها را با ColBERT بردارسازی و جستجو کنید

در نوت‌بوک بعدی (`colbert_farsi.ipynb`) همین روند را با یک مدل **چندزبانه** (Jina-ColBERT-v2) روی متن **فارسی** تکرار می‌کنیم.

</div>

In [30]:
# فراموش نکنید کلاینت را ببندید!
client.close()
